# schedule objective loss notify v1

目标：验证 `src/schedule` 的 model dashboard refresh 详细通知可以从统一 dashboard export 提取 `objective.loss.v1` 摘要，并在通知中报告 objective loss contract 状态。

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_objective_loss_notify_v1')
SOURCE_EXPORT = Path('research/model_matrix_objective_loss_generation_v1/dashboard_export.json')
EXPORT_PATH = CARD / 'dashboard_export.json'
LOCK_DIR = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'source_export': str(SOURCE_EXPORT),
    'source_export_exists': SOURCE_EXPORT.exists(),
}

{'card': 'research/schedule_objective_loss_notify_v1',
 'source_export': 'research/model_matrix_objective_loss_generation_v1/dashboard_export.json',
 'source_export_exists': True}

## 构造 schedule refresh

matrix runner 只确认 card 目录存在；dashboard exporter 从上一轮真实生成的 `dashboard_export.json` 读取 payload 并写到本实验目录。这样验证的是 schedule 通知 contract，而不是重新跑训练。

In [2]:
source_payload = json.loads(SOURCE_EXPORT.read_text(encoding='utf-8'))
assert source_payload['objective']['losses']['schema_version'] == 'objective.loss.v1'
assert source_payload['objective']['losses']['reward_surface'] == 'retired'

events: list[str] = []

def matrix_runner(**kwargs):
    Path(kwargs['card_dir']).mkdir(parents=True, exist_ok=True)
    return {'variant_count': len(source_payload.get('top_models', [])), 'status': 'reuse_dashboard_export_for_schedule_notify'}

def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(source_payload)
    payload['artifact_id'] = 'schedule_objective_loss_notify_v1'
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return payload

config = ModelDashboardPipelineConfig(
    name='model_dashboard_refresh',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=EXPORT_PATH,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf', 'gcn_mse'),
    notify_details=True,
)

result = run_model_dashboard_refresh_once(
    config,
    lock_dir=LOCK_DIR,
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)

assert result.status == 'success'
assert result.payload['objective_loss_schema_version'] == 'objective.loss.v1'
assert result.payload['objective_loss_reward_surface'] == 'retired'
assert result.payload['objective_loss_optimization_direction'] == 'minimize'
assert result.payload['objective_loss_top_component']
assert events and 'objective_loss=objective.loss.v1' in events[0]
assert 'reward=retired' in events[0]
{
    'status': result.status,
    'payload_loss_schema': result.payload['objective_loss_schema_version'],
    'payload_top_loss_component': result.payload['objective_loss_top_component'],
    'notification': events[0],
}

{'status': 'success',
 'payload_loss_schema': 'objective.loss.v1',
 'payload_top_loss_component': 'ranking_loss',
 'notification': 'model_dashboard_refresh:success | top_model=random_forest_clf | dashboard_manifest=analysis_manifest | analysis_preflight=missing_analysis_sidecars | attempts=1 | export=research/schedule_objective_loss_notify_v1/dashboard_export.json | top_failure=return_stability_lag@downtrend | top_review=weakness:block_or_rework@downtrend | backtest=backtest.config.v1 folds=5 slippage_bps=1.0 | regime_gap=downtrend:ridge_mse->random_forest_clf/spread=31.5881 | model_difference=segment_winner/promote_segment_candidate bull random_forest_clf->ridge_mse spread=30.1835 | promotion=promote_candidate:random_forest_clf blocked=0 | objective_loss=objective.loss.v1 top=ranking_loss:1 reward=retired | run_coverage=3/3 backtested=3/3 missing=0 | requested_models=3/32 missing=29 incomplete=0 | readiness=review pass=2 review=1 block=0'}

## 写出证据摘要

摘要保留通知文本和 payload 中的 objective loss 字段，方便中文 review。

In [3]:
summary = {
    'schema_version': 'research.schedule_objective_loss_notify.v1',
    'status': 'completed',
    'source_export': str(SOURCE_EXPORT),
    'dashboard_export': str(EXPORT_PATH),
    'notification_events': events,
    'objective_loss_summary_schema_version': result.payload['objective_loss_summary_schema_version'],
    'objective_loss_schema_version': result.payload['objective_loss_schema_version'],
    'objective_loss_metric_source_schema_version': result.payload['objective_loss_metric_source_schema_version'],
    'objective_loss_reward_surface': result.payload['objective_loss_reward_surface'],
    'objective_loss_optimization_direction': result.payload['objective_loss_optimization_direction'],
    'objective_loss_component_count': result.payload['objective_loss_component_count'],
    'objective_loss_top_component': result.payload['objective_loss_top_component'],
    'objective_loss_top_value': result.payload['objective_loss_top_value'],
    'conclusion': 'schedule detailed notification 已报告 objective.loss.v1、top loss component 和 retired reward surface。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'notification_events.json').write_text(json.dumps({'events': events}, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.schedule_objective_loss_notify.v1',
 'status': 'completed',
 'source_export': 'research/model_matrix_objective_loss_generation_v1/dashboard_export.json',
 'dashboard_export': 'research/schedule_objective_loss_notify_v1/dashboard_export.json',
 'notification_events': ['model_dashboard_refresh:success | top_model=random_forest_clf | dashboard_manifest=analysis_manifest | analysis_preflight=missing_analysis_sidecars | attempts=1 | export=research/schedule_objective_loss_notify_v1/dashboard_export.json | top_failure=return_stability_lag@downtrend | top_review=weakness:block_or_rework@downtrend | backtest=backtest.config.v1 folds=5 slippage_bps=1.0 | regime_gap=downtrend:ridge_mse->random_forest_clf/spread=31.5881 | model_difference=segment_winner/promote_segment_candidate bull random_forest_clf->ridge_mse spread=30.1835 | promotion=promote_candidate:random_forest_clf blocked=0 | objective_loss=objective.loss.v1 top=ranking_loss:1 reward=retired | run_coverage=3

## 中文结论

- 做了什么：验证 `run_model_dashboard_refresh_once(..., notify_details=True)` 会从 dashboard export 的 `objective.loss.v1` 生成通知摘要。
- 核心指标：通知包含 `objective_loss=objective.loss.v1`、top loss component、top loss value 和 `reward=retired`；payload 保留 `dashboard.objective_loss_summary.v1`、`optimization_direction=minimize` 和 loss component count。
- regime 差异：本实验只验证 schedule 通知通路；regime/window 表现仍由 dashboard export 和 analysis sidecar 承接。
- 是否晋升：这是 `src/schedule` 框架能力晋升，不是策略晋升。
- 风险：通知只放最小摘要，完整 loss components 仍在 dashboard export/view model；后续若做逐模型 loss，需要扩展 payload 格式。
- 下一步：可以把 paper execution 或 production health gate 也纳入 objective loss/readiness 检查。